# Qwen3Loop-0.6B Tool-Calling DPO round4-bestofn (v38 + beta0.1 + sft_anchor + loop_aux)
Policy/ref = v38 (HF, ancora fixa). Pares R2 (256, sem placeholder). DPO beta 0.2 + CE auxiliar nos snapshots do loop (loop_aux 0.2).

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# 2. Deps
!pip install -q transformers datasets accelerate safetensors huggingface_hub


In [ ]:
# 3. Descobrir engine + script + pares (dataset) e baixar v38 do HF
import os, shutil
def find(base_list, fname):
    for base in base_list:
        for root, dirs, files in os.walk(base):
            if fname in files:
                return os.path.join(root, fname)
    return None
bases = ['/kaggle/input']
engine_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'modeling_qwen3loop.py' in files and 'configuration_qwen3loop.py' in files:
        try:
            mk = open(os.path.join(root, 'modeling_qwen3loop.py')).read()
        except Exception:
            continue
        if 'filtered_kwargs' in mk:
            engine_dir = root
            break
assert engine_dir, 'engine COM marker nao encontrada'
os.makedirs('/kaggle/working/engine', exist_ok=True)
shutil.copytree(engine_dir, '/kaggle/working/engine/qwen3loop_python', dirs_exist_ok=True)
shutil.copytree(engine_dir, '/kaggle/working/qwen3loop_python', dirs_exist_ok=True)
import sys
sys.path.insert(0, '/kaggle/working')
sys.path.insert(0, '/kaggle/working/engine')
script_path = find(bases, 'train_dpo_tool.py')
chosen_path = find(bases, 'dpo_r4c_chosen.jsonl')
rejected_path = find(bases, 'dpo_r4c_rejected.jsonl')
print('engine:', engine_dir)
print('script:', script_path)
print('chosen:', chosen_path)
print('rejected:', rejected_path)
assert all([script_path, chosen_path, rejected_path])
from huggingface_hub import snapshot_download
model_path = snapshot_download('Lordnyx/qwen3loop-0.6b-tool-calling-sft-v38',
    allow_patterns=['model.safetensors', 'config.json', 'tokenizer*',
                    'chat_template.jinja', 'generation_config.json'])
print('v38:', model_path)


In [ ]:
# 4. DPO round3 (1 GPU, bf16). ETA ~2-3h (pares longos)
output_dir = '/kaggle/working/qwen3loop_tool_bon_r4'
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && python {script_path} \
    --pairs {chosen_path} \
    --rejected {rejected_path} \
    --model_path {model_path} \
    --output_dir {output_dir} \
    --lr 5e-7 --epochs 2 --beta 0.1 --loop_aux 0.2 --loop_aux_snaps 1 --sft_anchor 0.5


In [ ]:
# 5. Confere saida
import os
final = '/kaggle/working/qwen3loop_tool_dpo_v3/model.safetensors'
print('exists:', os.path.exists(final))
if os.path.exists(final):
    print('bytes:', os.path.getsize(final))
